# Instructblip Flan T5 Xl Nocaps Baseline

This notebook was reorganized for the GitHub reproducibility package.
Original file: `NoCaps-Baseline/InstructBLIP-NoCaps.ipynb`.

**Security note:** hard-coded Hugging Face tokens were removed. Use interactive login or environment variables instead.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# InstructBLIP için gerekli kütüphaneler
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q accelerate bitsandbytes
!pip install -q requests pillow

print("✅ Kurulumlar tamamlandı. Devam edebilirsin.")

In [ ]:
import torch
from transformers import InstructBlipProcessor, InstructBlipForConditionalGeneration
from PIL import Image
import os
import json
from tqdm import tqdm

# --- AYARLAR ---
MODEL_ID = "Salesforce/instructblip-flan-t5-xl"
GT_PATH = "/content/drive/MyDrive/datasets/nocaps/nocaps_val_4500_captions_domain_norm.json"
IMG_DIR = "/content/drive/MyDrive/datasets/nocaps/images_val_hf"
OUTPUT_FILE = "instructblip_nocaps_results.json"

# InstructBLIP için Talimat (Instruction)
# Modelin ne yapacağını söylüyoruz.
PROMPT = "A short image description."

# Dosya Kontrolü
if not os.path.exists(GT_PATH):
    raise FileNotFoundError(f"❌ JSON bulunamadı: {GT_PATH}")

# 1. Modeli Yükle
print(f"⏳ {MODEL_ID} yükleniyor...")
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = InstructBlipProcessor.from_pretrained(MODEL_ID)
model = InstructBlipForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto" # GPU belleğini otomatik yönetir
)
model.eval()
print("✅ InstructBLIP Modeli Hazır!")


In [ ]:
# 2. Tahmin Döngüsü
with open(GT_PATH, "r", encoding="utf-8") as f:
    nocaps_gt = json.load(f)

results = []
print(f"🚀 {len(nocaps_gt['images'])} görsel için InstructBLIP çalışıyor...")

for img_info in tqdm(nocaps_gt['images']):
    image_path = os.path.join(IMG_DIR, img_info['file_name'])

    if not os.path.exists(image_path):
        continue

    try:
        image = Image.open(image_path).convert("RGB")

        # Girdi Hazırlığı (Resim + Prompt)
        inputs = processor(images=image, text=PROMPT, return_tensors="pt").to(device, torch.float16)

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=80,
                num_beams=5,
                do_sample=False,
                min_length=1,
                repetition_penalty=1.5,
                length_penalty=1.0,
                temperature=1,
            )

        # Çıktıyı Metne Dönüştür
        caption = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

        results.append({
            "image_id": img_info['id'],
            "caption": caption
        })

    except Exception as e:
        print(f"Hata ({img_info['file_name']}): {e}")

# Kaydet
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\n💾 Sonuçlar kaydedildi: {OUTPUT_FILE}")

In [ ]:
import os
import sys
import json

# --- 1. ORTAM HAZIRLIĞI ---
print("🛠️ Skorlama ortamı hazırlanıyor...")

# Java
os.system("apt-get install -y openjdk-8-jdk-headless -qq > /dev/null")
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

# PyCocoEvalCap
if not os.path.exists("pycocoevalcap"):
    os.system("git clone https://github.com/salaniz/pycocoevalcap.git")

# Tokenizer (Manuel İndir)
target_dir = "pycocoevalcap/tokenizer"
target_file = os.path.join(target_dir, "stanford-corenlp-3.4.1.jar")
os.makedirs(target_dir, exist_ok=True)

if not os.path.exists(target_file):
    print("⬇️ Tokenizer dosyası indiriliyor...")
    os.system(f"wget -q https://repo1.maven.org/maven2/edu/stanford/nlp/stanford-corenlp/3.4.1/stanford-corenlp-3.4.1.jar -O {target_file}")

sys.path.append(os.path.abspath("pycocoevalcap"))

# --- 2. HESAPLAMA ---
from tokenizer.ptbtokenizer import PTBTokenizer
from bleu.bleu import Bleu
from rouge.rouge import Rouge
from cider.cider import Cider
from meteor.meteor import Meteor

# Dosya Yolları
GT_PATH = "/content/drive/MyDrive/datasets/nocaps/nocaps_val_4500_captions_domain_norm.json"
RES_PATH = "instructblip_nocaps_results.json" # <-- InstructBLIP Sonuçları

if not os.path.exists(RES_PATH):
    raise FileNotFoundError("❌ Sonuç dosyası bulunamadı! Önceki adımı çalıştırdın mı?")

print(f"\n📊 InstructBLIP Skorları Hesaplanıyor...")

coco = json.load(open(GT_PATH))
cocoRes = json.load(open(RES_PATH))

# Formatlama
gts = {ann['image_id']: [] for ann in coco['annotations']}
for ann in coco['annotations']: gts[ann['image_id']].append(ann)
res = {ann['image_id']: [ann] for ann in cocoRes}

# Domain ID'leri
ids_in   = [img['id'] for img in coco['images'] if img['domain_norm'] == 'in']
ids_near = [img['id'] for img in coco['images'] if img['domain_norm'] == 'near']
ids_out  = [img['id'] for img in coco['images'] if img['domain_norm'] == 'out']
ids_all  = [img['id'] for img in coco['images']]

def evaluate_manual(img_ids, title):
    if not img_ids: return
    print(f"\n{'='*10} {title} ({len(img_ids)}) {'='*10}")

    gts_curr = {i: gts[i] for i in img_ids if i in gts}
    res_curr = {i: res[i] for i in img_ids if i in res}

    tokenizer = PTBTokenizer()
    gts_tok = tokenizer.tokenize(gts_curr)
    res_tok = tokenizer.tokenize(res_curr)

    scorers = [
        (Bleu(4), ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4"]),
        (Meteor(), "METEOR"),
        (Rouge(), "ROUGE_L"),
        (Cider(), "CIDEr")
    ]

    for scorer, method in scorers:
        try:
            score, scores = scorer.compute_score(gts_tok, res_tok)
            if isinstance(method, list):
                for m, s in zip(method, score): print(f"{m:10s}: {s:.3f}")
            else:
                print(f"{method:10s}: {score:.3f}")
        except Exception as e:
            print(f"Hata: {e}")

evaluate_manual(ids_in, "IN-DOMAIN")
evaluate_manual(ids_near, "NEAR-DOMAIN")
evaluate_manual(ids_out, "OUT-OF-DOMAIN")
evaluate_manual(ids_all, "OVERALL")

In [ ]:
def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"🤖 Model: {MODEL_ID}")
    print(f"🔢 Toplam Parametre: {total_params:,}")
    print(f"📉 Eğitilebilir Parametre: {trainable_params:,}")
    print(f"📊 Milyar Cinsinden: {total_params / 1e9:.2f}B")

# Hesapla ve Yazdır
count_parameters(model)